# Module 7: State Space, Unobserved Components and ETS

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Prairie County never submitted three months in the spring of 2022. Intermediate
[Module 2](../../Intermediate/Module_02_Building_An_Honest_Calendar.md) said to
leave those months missing rather than filling them with zero, and that was the
right advice.

It also makes the series unusable by most of the methods in this series. This
module is about the family that can cope, and about how the others fail, which
is more interesting than it sounds because **two of them fail without saying
so**.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]          # never fit on unfinished months


CALENDAR = pd.period_range("2019-01", "2026-04", freq="M").to_timestamp()


def counts(agency_id):
    """Monthly counts on a complete calendar, so a gap stays visible as missing."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series(d["n_uof"].values, dtype=float,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


print(f"{final['agency_id'].nunique()} agencies, {final['year_month'].nunique()} months")

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox


def ljung(resid, lag=12):
    return float(acorr_ljungbox(resid, lags=[lag], return_df=True)["lb_pvalue"].iloc[0])

p = counts("A009")                         # Prairie County, three months missing
gap = p.index[p.isna()]
print(f"{len(p)} months, {int(p.isna().sum())} missing: "
      f"{[str(k)[:7] for k in gap]}")

lp = np.log(p + 0.5)                       # the half keeps the zeros finite
lp.index.freq = "MS"

## 2. Hand the same gapped series to three methods

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import STL

state_space = SARIMAX(lp, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
ss_fitted = state_space.get_prediction().predicted_mean

holt = ExponentialSmoothing(lp, trend="add", seasonal="add", seasonal_periods=12,
                            initialization_method="estimated").fit()
hw_fitted = pd.Series(np.asarray(holt.fittedvalues), index=lp.index)

stl_trend = pd.Series(np.asarray(STL(lp, period=12, robust=True).fit().trend),
                      index=lp.index)

for name, v in [("state space (SARIMAX)", ss_fitted),
                ("Holt Winters", hw_fitted),
                ("STL", stl_trend)]:
    bad = int(np.isnan(np.asarray(v)).sum())
    print(f"  {name:24s} unusable fitted values: {bad:2d} of {len(v)}")

**None of the three raised an error.** One of them worked.

State space produced a usable value for every month. Holt Winters produced
nothing for about half the series. STL produced nothing at all, for every
month, having been asked to smooth a series with holes in it.

In [ ]:
print("Holt Winters forecast for the next six months:")
print(" ", np.round(np.asarray(holt.forecast(6)), 3))
print("\nstate space forecast for the next six months, back on the count scale:")
print(" ", np.round(np.exp(state_space.forecast(6).values) - 0.5, 2))

An all missing forecast, returned without complaint. In a pipeline that writes
to a dashboard, that is a blank chart nobody can explain, and the cause is
three months three years earlier.

## 3. Why the state space version can do it

A state space model separates what it believes about the world from what it
observed. The **Kalman filter** carries an estimate of the state forward month
by month, updating it whenever an observation arrives.

When a month is missing there is simply no update. The filter propagates the
state, widens its uncertainty, and continues. Nothing special is required and
nothing is imputed.

In [ ]:
est = np.exp(ss_fitted.loc[gap].values) - 0.5
print("what the filter estimates for the three months nobody reported:")
for k, v in zip(gap, est):
    print(f"  {str(k)[:7]}  {v:.2f} incidents")
print(f"\nthe agency's average over the rest of the series: "
      f"{np.nanmean(p.values):.2f}")

**These are estimates, not data.** They belong in a model, and they do not
belong in a table of what happened. The reporting rule from Intermediate Module
2 does not change: the months were never submitted and any published count for
them is an estimate that must be labelled.

## 4. The tempting shortcut, and why it is wrong

The obvious alternative is to delete the three months and carry on.

In [ ]:
dropped = lp.dropna()
print(f"  length after dropping: {len(dropped)}")
print(f"  index frequency: {dropped.index.freq}")
print(f"  the month after 2022-02 is now {str(dropped.index[dropped.index.get_loc(pd.Timestamp('2022-02-01')) + 1])[:7]}")

The index loses its regular frequency, and **the model is now told that June
follows February by one step.** Every lag in the model is wrong from that point
on, the seasonal period no longer lines up with the calendar, and nothing warns
you.

Deleting is worse than leaving the gap, because leaving the gap at least gives
the honest methods a chance to handle it.

## 5. Unobserved components: the same machinery, readable parts

A SARIMA describes a series through its correlations. An **unobserved
components** model describes it as a sum of pieces you can name and plot:
a level, a slope, a seasonal component, and noise. Same state space machinery,
same tolerance of gaps, interpretable output.

In [ ]:
from statsmodels.tsa.statespace.structural import UnobservedComponents

uc = UnobservedComponents(lp, level="local linear trend", seasonal=12).fit(disp=False)

print(f"  variance of the level shocks   {uc.params['sigma2.level']:.5f}")
print(f"  variance of the slope shocks   {uc.params['sigma2.trend']:.5f}")
print(f"  variance of the seasonal shocks{uc.params['sigma2.seasonal']:.5f}")
print(f"  variance of the noise          {uc.params['sigma2.irregular']:.5f}")
print(f"\n  AIC {uc.aic:.1f}   against the SARIMA's {state_space.aic:.1f}")

Read the variances as a question about what is allowed to move. A near zero
seasonal variance says the seasonal pattern is fixed; a large one says it
drifts. The same information is inside a SARIMA, and it is not written anywhere
you can read it.

**Use unobserved components when someone will ask what the model thinks the
trend is.** Use SARIMA when forecast accuracy is the only deliverable.

## 6. Where Prophet fits

Prophet is widely used and worth a paragraph rather than a section.

It fits a piecewise linear trend with automatically placed changepoints plus
seasonal terms, and its appeal is that it produces a plausible chart with no
decisions. For public safety data that appeal is also the problem.

| Consideration | For this kind of data |
|---|---|
| Automatic changepoints | it will place breaks where none occurred, which matters when the question **is** whether a break occurred |
| Designed for high frequency business data | monthly police counts have neither the volume nor the granularity it was built for |
| Handles gaps and outliers by default | convenient, and it hides exactly the data problems Beginner Topic 19 says to look for |
| Maintenance | development has been largely dormant for several years |

**A reasonable default is not to use it here.** If it is used, report a SARIMA
or unobserved components fit alongside it, and check that the changepoints it
chose correspond to something that actually happened.

## 7. Which member of the family

| Situation | Reach for |
|---|---|
| Missing months | state space, any of them; not STL, not Holt Winters |
| Someone will ask what the trend is | unobserved components |
| Forecast accuracy is the only goal | SARIMA, and check it against the baseline |
| Small counts with zeros | a count model, [Module 9](Module_09_Rare_Events.ipynb) |
| An intervention with a known date | [Module 11](Module_11_Interrupted_Time_Series.ipynb) |
| You want a chart with no decisions | reconsider; that is the appeal that causes the trouble |

## Exercise

Confirm that the gap is what breaks Holt Winters, rather than something else
about Prairie County, by filling the three months and refitting.

In [ ]:
# Fill in the blank, then run.
FILL_THE_GAP = None        # try True

if FILL_THE_GAP is not None:
    test_series = lp.interpolate() if FILL_THE_GAP else lp
    h = ExponentialSmoothing(test_series, trend="add", seasonal="add",
                             seasonal_periods=12,
                             initialization_method="estimated").fit()
    bad = int(np.isnan(np.asarray(h.fittedvalues)).sum())
    label = "gap filled by interpolation" if FILL_THE_GAP else "gap left missing"
    print(f"  {label:32s} unusable fitted values: {bad} of {len(test_series)}")
    print(f"  {'':32s} forecast usable: "
          f"{not bool(np.isnan(np.asarray(h.forecast(3))).any())}")
else:
    print("Set FILL_THE_GAP above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
FILL_THE_GAP = True
```

With the gap filled, Holt Winters works perfectly: every fitted value is
usable and the forecast comes back. The gap was the whole problem.

**That does not make interpolation the right answer.** What has happened is
that three invented numbers are now indistinguishable from the 85 real ones,
and every figure the model produces afterwards treats them as observations.
The estimate may be reasonable; the loss of the distinction is not.

The two defensible routes are the ones in section 2 and section 5: use a method
that handles the gap natively and knows those months were unobserved, or
interpolate deliberately, **label the interpolated months everywhere they
appear**, and report how much the results change with and without them.

What is not defensible is interpolating because a library complained, and then
forgetting.

</details>

---

**Next:** Part III, on counts, rare events and many agencies at once.

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*